# f6_m04a_stress.ipynb
**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 6 — Interpretabilidad y Evaluación Final |
| **Módulo** | M04a — Stress Testing |

---

## 🎯 Qué hace

Evalúa la robustez del modelo ganador (leído de `metricas_modelo.json`) ante
perturbaciones controladas. Simula 3 escenarios adversos:
- **Ruido gaussiano** en features numéricas (5%–50% del std)
- **Valores extremos** (outliers a media ± 3σ, 1%–30% obs)
- **Features ausentes** (imputadas con mediana, 1–5 features top SHAP)

Mide cómo cambian AUC y F1 bajo cada perturbación.

## 📋 Requisitos

- `data/06_evaluacion/metricas_modelo.json` — fuente única de verdad del ganador
- `data/05_modelado/X_test_prep.parquet`
- `data/05_modelado/y_test.parquet`
- `data/05_modelado/models/<modelo_ganador>.pkl` (dinámico)
- `results/fase6/shap_importancia_comparativa.parquet` (opcional, para missing)

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `results/fase6/stress_resultados.parquet` | Resultados de los 3 escenarios |
| `results/fase6/stress_ruido.png` | AUC/F1 bajo ruido gaussiano |
| `results/fase6/stress_outliers.png` | AUC/F1 bajo outliers |
| `results/fase6/stress_missing.png` | AUC/F1 bajo features ausentes |
| `results/fase6/stress_degradacion_relativa.png` | 🏆 % pérdida vs baseline |
| `docs/html/fase6/m04a_stress.html` | Informe HTML |

## 🔄 Flujo

```
metricas_modelo.json → ganador dinámico
X_test_prep + y_test + modelo
    ↓ Baseline AUC/F1
    ↓ Stress 1: Ruido gaussiano
    ↓ Stress 2: Outliers (media ± 3σ)
    ↓ Stress 3: Missing top features SHAP
    ↓ Gráficos absolutos + degradación relativa
    → stress_resultados.parquet + m04a_stress.html
```

## ➡️ Siguiente

`f6_m04b_calibracion.ipynb` — calibración de probabilidades


In [1]:
# ============================================================
# CELDA 1: CONFIGURACIÓN DE RUTAS
# ROOT detectado subiendo niveles hasta encontrar src/
# ============================================================
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

DIR_DATA    = ROOT / 'data' / '05_modelado'
DIR_MODELS  = ROOT / 'data' / '05_modelado' / 'models'
DIR_RESULTS = ROOT / 'results' / 'fase6'
DIR_HTML    = ROOT / 'docs' / 'html' / 'fase6'
DIR_RESULTS.mkdir(parents=True, exist_ok=True)
DIR_HTML.mkdir(parents=True, exist_ok=True)

# JSON del ganador dinámico
RUTA_JSON = ROOT / 'data' / '06_evaluacion' / 'metricas_modelo.json'

print(f'ROOT:        {ROOT}')
print(f'DIR_MODELS:  {DIR_MODELS}')
print(f'DIR_RESULTS: {DIR_RESULTS}')
print(f'RUTA_JSON:   {RUTA_JSON}')

ROOT:        C:\FF\AU_UJI_v2
DIR_MODELS:  C:\FF\AU_UJI_v2\data\05_modelado\models
DIR_RESULTS: C:\FF\AU_UJI_v2\results\fase6
RUTA_JSON:   C:\FF\AU_UJI_v2\data\06_evaluacion\metricas_modelo.json


In [2]:
# ============================================================
# CELDA 2: IMPORTS Y CARGA DEL GANADOR DINÁMICO
# Sistema dinámico: el ganador se lee de metricas_modelo.json
# ============================================================
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.metrics import roc_auc_score, f1_score
from src.html.render import render_pagina
from src.config_entorno import NOMBRES_LEGIBLES_FEATURES

plt.rcParams['figure.dpi'] = 120
RNG = np.random.default_rng(42)

def nombre_legible(f):
    return NOMBRES_LEGIBLES_FEATURES.get(f, f.replace('_', ' '))

# Cargar metadatos del modelo ganador (sistema dinámico)
assert RUTA_JSON.exists(), f'❌ No encontrado: {RUTA_JSON}'
with open(RUTA_JSON, encoding='utf-8') as f:
    meta_json = json.load(f)

nombre_ganador_pkl = meta_json['modelo_pkl']
nombre_ganador     = meta_json['modelo_nombre']
familia_ganador    = meta_json['modelo_familia']

print('Imports OK.')
print(f'Modelo ganador: {nombre_ganador} ({familia_ganador})')
print(f'PKL:            {nombre_ganador_pkl}')

Imports OK.
Modelo ganador: LightGBM (Gradient Boosting)
PKL:            LightGBM__none.pkl


In [3]:
# ============================================================
# CELDA 3: CARGAR DATOS Y MODELO + BASELINE
# Pipeline completo para predict_proba.
# ============================================================
X_test_prep    = pd.read_parquet(DIR_DATA / 'X_test_prep.parquet')
y_test         = pd.read_parquet(DIR_DATA / 'y_test.parquet').squeeze()
modelo_ganador = joblib.load(DIR_MODELS / nombre_ganador_pkl)  # carga dinámica

y_true = y_test.values.ravel()
features_num = X_test_prep.select_dtypes(include=[np.number]).columns.tolist()
features_cat = X_test_prep.select_dtypes(exclude=[np.number]).columns.tolist()

# Baseline
y_prob_base = modelo_ganador.predict_proba(X_test_prep)[:, 1]
y_pred_base = (y_prob_base >= 0.5).astype(int)
auc_base    = roc_auc_score(y_true, y_prob_base)
f1_base     = f1_score(y_true, y_pred_base)

print(f'Modelo:               {nombre_ganador}')
print(f'Baseline — AUC: {auc_base:.4f} | F1: {f1_base:.4f}')
print(f'Features numéricas:   {len(features_num)}')
print(f'Features categóricas: {len(features_cat)}')

Modelo:               LightGBM
Baseline — AUC: 0.9564 | F1: 0.8334
Features numéricas:   27
Features categóricas: 0


In [4]:
# ============================================================
# CELDA 4: STRESS TEST 1 — RUIDO GAUSSIANO
# Ruido proporcional a la desviación estándar de cada feature.
# Niveles: 5%, 10%, 20%, 30%, 50% del std.
# Simula errores de medición o datos de baja calidad.
# ============================================================
niveles_ruido = [0.05, 0.10, 0.20, 0.30, 0.50]
resultados_ruido = []

stds = X_test_prep[features_num].std()

for nivel in niveles_ruido:
    X_pert = X_test_prep.copy()
    ruido  = RNG.normal(0, nivel, size=X_pert[features_num].shape) * stds.values
    X_pert[features_num] = X_pert[features_num] + ruido

    y_prob = modelo_ganador.predict_proba(X_pert)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    auc    = roc_auc_score(y_true, y_prob)
    f1     = f1_score(y_true, y_pred)
    resultados_ruido.append({'nivel': nivel, 'auc': auc, 'f1': f1})
    print(f'Ruido {nivel*100:.0f}% std — AUC: {auc:.4f} | F1: {f1:.4f}')

df_ruido = pd.DataFrame(resultados_ruido)

Ruido 5% std — AUC: 0.9526 | F1: 0.8255


Ruido 10% std — AUC: 0.9468 | F1: 0.8122


Ruido 20% std — AUC: 0.9342 | F1: 0.7807


Ruido 30% std — AUC: 0.9151 | F1: 0.7546


Ruido 50% std — AUC: 0.8852 | F1: 0.7149


In [5]:
# ============================================================
# CELDA 5: STRESS TEST 2 — VALORES EXTREMOS (OUTLIERS)
# Sustituimos un porcentaje de observaciones por valores extremos
# (media ± 3*std). Simula errores de entrada o casos atípicos.
# ============================================================
pcts_outliers = [0.01, 0.05, 0.10, 0.20, 0.30]
resultados_outliers = []

medias = X_test_prep[features_num].mean()

for pct in pcts_outliers:
    X_pert = X_test_prep.copy()
    n_pert = int(len(X_pert) * pct)
    idx_pert = RNG.choice(len(X_pert), size=n_pert, replace=False)

    # Valores extremos: media + 3*std con signo aleatorio
    signos = RNG.choice([-1, 1], size=(n_pert, len(features_num)))
    X_pert.iloc[idx_pert, X_pert.columns.get_indexer(features_num)] = (
        medias.values + signos * 3 * stds.values
    )

    y_prob = modelo_ganador.predict_proba(X_pert)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    auc    = roc_auc_score(y_true, y_prob)
    f1     = f1_score(y_true, y_pred)
    resultados_outliers.append({'pct': pct, 'auc': auc, 'f1': f1})
    print(f'Outliers {pct*100:.0f}% obs — AUC: {auc:.4f} | F1: {f1:.4f}')

df_outliers = pd.DataFrame(resultados_outliers)

Outliers 1% obs — AUC: 0.9534 | F1: 0.8314


Outliers 5% obs — AUC: 0.9284 | F1: 0.8093
Outliers 10% obs — AUC: 0.9018 | F1: 0.7839


Outliers 20% obs — AUC: 0.8436 | F1: 0.7260


Outliers 30% obs — AUC: 0.8097 | F1: 0.6882


In [6]:
# ============================================================
# CELDA 6: STRESS TEST 3 — FEATURES AUSENTES (MISSING)
# Sustituimos features con la mediana (imputación simple).
# Probamos eliminar de 1 a 5 features importantes según SHAP.
# Si no existe el ranking SHAP, usamos las primeras features numéricas.
# ============================================================
RUTA_SHAP = DIR_RESULTS / 'shap_importancia_comparativa.parquet'
if RUTA_SHAP.exists():
    df_shap_imp  = pd.read_parquet(RUTA_SHAP)
    features_top = df_shap_imp.nsmallest(5, 'rank_medio').index.tolist()
    features_top = [f for f in features_top if f in features_num]
else:
    features_top = features_num[:5]

resultados_missing = []
medianas = X_test_prep[features_num].median()

for n_missing in range(1, len(features_top) + 1):
    feats_eliminar = features_top[:n_missing]
    X_pert = X_test_prep.copy()
    X_pert[feats_eliminar] = medianas[feats_eliminar].values

    y_prob = modelo_ganador.predict_proba(X_pert)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    auc    = roc_auc_score(y_true, y_prob)
    f1     = f1_score(y_true, y_pred)
    resultados_missing.append({
        'n_features': n_missing,
        'features':   ', '.join(feats_eliminar),
        'auc':        auc,
        'f1':         f1
    })
    print(f'Missing {n_missing} feat — AUC: {auc:.4f} | F1: {f1:.4f} | {feats_eliminar}')

df_missing = pd.DataFrame(resultados_missing)

Missing 1 feat — AUC: 0.9074 | F1: 0.7073 | ['cred_superados_anio_1er']
Missing 2 feat — AUC: 0.9052 | F1: 0.6780 | ['cred_superados_anio_1er', 'n_anios_trabajando']


Missing 3 feat — AUC: 0.8667 | F1: 0.6044 | ['cred_superados_anio_1er', 'n_anios_trabajando', 'n_anios_beca']
Missing 4 feat — AUC: 0.8846 | F1: 0.5689 | ['cred_superados_anio_1er', 'n_anios_trabajando', 'n_anios_beca', 'anios_sin_beca']
Missing 5 feat — AUC: 0.8377 | F1: 0.4565 | ['cred_superados_anio_1er', 'n_anios_trabajando', 'n_anios_beca', 'anios_sin_beca', 'cred_repetidos']


In [7]:
# ============================================================
# CELDA 7: GUARDAR RESULTADOS Y GRÁFICOS
# Paleta UJI 2026 (alineada con config_app.py):
#   azul  (#1e4d8c) = ruido / AUC
#   rojo  (#dc2626) = outliers / F1
#   ámbar (#f59e0b) = missing / umbral 5%
# ============================================================
COLOR_RUIDO    = '#1e4d8c'  # COLORES["primario"]
COLOR_OUTLIERS = '#dc2626'  # COLORES["abandono"]
COLOR_MISSING  = '#f59e0b'  # COLORES["advertencia"]
COLOR_F1       = '#dc2626'  # rojo abandono
COLOR_AUC      = '#1e4d8c'  # azul primario

# Guardar todos los resultados
df_stress = pd.concat([
    df_ruido.assign(test='ruido'),
    df_outliers.assign(test='outliers'),
    df_missing.assign(test='missing')
], ignore_index=True)
df_stress.to_parquet(DIR_RESULTS / 'stress_resultados.parquet')

# --- Gráfico ruido ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metrica in zip(axes, ['auc', 'f1']):
    baseline = auc_base if metrica == 'auc' else f1_base
    ax.plot([n*100 for n in df_ruido['nivel']], df_ruido[metrica],
            marker='o', color=COLOR_RUIDO, linewidth=2)
    ax.axhline(baseline, color='gray', linestyle='--', linewidth=1,
               label=f'Baseline ({baseline:.3f})')
    ax.set_xlabel('Nivel de ruido (% std)')
    ax.set_ylabel(metrica.upper())
    ax.set_title(f'{metrica.upper()} bajo ruido gaussiano')
    ax.legend(fontsize=9)
plt.suptitle(
    f'Stress test ({nombre_ganador}) — Ruido gaussiano en features numéricas',
    fontsize=12
)
plt.tight_layout()
ruta_ruido = DIR_RESULTS / 'stress_ruido.png'
plt.savefig(ruta_ruido, dpi=120, bbox_inches='tight')
plt.close()

# --- Gráfico outliers ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metrica in zip(axes, ['auc', 'f1']):
    baseline = auc_base if metrica == 'auc' else f1_base
    ax.plot([p*100 for p in df_outliers['pct']], df_outliers[metrica],
            marker='o', color=COLOR_OUTLIERS, linewidth=2)
    ax.axhline(baseline, color='gray', linestyle='--', linewidth=1,
               label=f'Baseline ({baseline:.3f})')
    ax.set_xlabel('% observaciones con outliers')
    ax.set_ylabel(metrica.upper())
    ax.set_title(f'{metrica.upper()} bajo outliers')
    ax.legend(fontsize=9)
plt.suptitle(
    f'Stress test ({nombre_ganador}) — Valores extremos (media ± 3σ)',
    fontsize=12
)
plt.tight_layout()
ruta_outliers = DIR_RESULTS / 'stress_outliers.png'
plt.savefig(ruta_outliers, dpi=120, bbox_inches='tight')
plt.close()

# --- Gráfico missing ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metrica in zip(axes, ['auc', 'f1']):
    baseline = auc_base if metrica == 'auc' else f1_base
    ax.plot(df_missing['n_features'], df_missing[metrica],
            marker='o', color=COLOR_MISSING, linewidth=2)
    ax.axhline(baseline, color='gray', linestyle='--', linewidth=1,
               label=f'Baseline ({baseline:.3f})')
    ax.set_xlabel('Nº features top eliminadas (imputadas con mediana)')
    ax.set_ylabel(metrica.upper())
    ax.set_title(f'{metrica.upper()} bajo missing top features')
    ax.legend(fontsize=9)
plt.suptitle(
    f'Stress test ({nombre_ganador}) — Features ausentes (imputación con mediana)',
    fontsize=12
)
plt.tight_layout()
ruta_missing = DIR_RESULTS / 'stress_missing.png'
plt.savefig(ruta_missing, dpi=120, bbox_inches='tight')
plt.close()

print('Gráficos absolutos guardados.')

# --- Gráfico degradación relativa (% pérdida vs baseline) — CUM LAUDE ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (df_plot, x_col, x_label, titulo_plot) in zip(axes, [
    (df_ruido,    [n*100 for n in df_ruido['nivel']],   'Nivel ruido (% std)', 'Ruido gaussiano'),
    (df_outliers, [p*100 for p in df_outliers['pct']], '% obs con outliers',   'Valores extremos'),
]):
    pct_auc = (auc_base - df_plot['auc']) / auc_base * 100
    pct_f1  = (f1_base  - df_plot['f1'])  / f1_base  * 100
    ax.plot(x_col, pct_auc, marker='o', color=COLOR_AUC, linewidth=2, label='% pérdida AUC')
    ax.plot(x_col, pct_f1,  marker='s', color=COLOR_F1,  linewidth=2, label='% pérdida F1')
    ax.axhline(5, color=COLOR_MISSING, linestyle='--', linewidth=1, alpha=0.7, label='Umbral 5%')
    ax.set_xlabel(x_label)
    ax.set_ylabel('% degradación vs baseline')
    ax.set_title(f'Degradación relativa — {titulo_plot}')
    ax.legend(fontsize=9)
    ax.set_ylim(bottom=0)

plt.suptitle(
    f'Fase 6 ({nombre_ganador}) — Degradación relativa (% pérdida respecto al baseline)',
    fontsize=12
)
plt.tight_layout()
ruta_degradacion = DIR_RESULTS / 'stress_degradacion_relativa.png'
plt.savefig(ruta_degradacion, dpi=120, bbox_inches='tight')
plt.close()
print('✅ Gráfico degradación relativa guardado.')

Gráficos absolutos guardados.


✅ Gráfico degradación relativa guardado.


In [8]:
# ============================================================
# CELDA 8: GENERAR HTML
# render_pagina — estándar del proyecto.
# Incluye bloque Wilcoxon dinámico para justificar el modelo ganador.
# Paleta UJI 2026 (alineada con config_app.py).
# ============================================================
import base64
from src.html.wilcoxon_block import bloque_wilcoxon_html

COLOR_PRIMARIO = '#1e4d8c'
COLOR_ABANDONO = '#dc2626'

def img_b64(ruta) -> str:
    if not ruta or not Path(ruta).exists():
        return ''
    with open(ruta, 'rb') as fh:
        return base64.b64encode(fh.read()).decode()

def bloque_imagen(b64: str, titulo: str, caption: str) -> str:
    if not b64:
        return f'<p style="color:{COLOR_ABANDONO}">⚠️ Imagen no disponible: {titulo}</p>'
    return (
        '<div style="margin:24px 0">'
        f'<h3 style="color:#2d3748;font-size:15px">{titulo}</h3>'
        f'<img src="data:image/png;base64,{b64}" '
        'style="max-width:100%;border-radius:6px;box-shadow:0 2px 8px rgba(0,0,0,.1)">'
        f'<p style="color:#718096;font-size:12px;margin-top:6px">{caption}</p>'
        '</div>'
    )

# Tabla resumen de degradación máxima
degradacion_auc_ruido    = auc_base - df_ruido['auc'].min()
degradacion_f1_ruido     = f1_base  - df_ruido['f1'].min()
degradacion_auc_outliers = auc_base - df_outliers['auc'].min()
degradacion_f1_outliers  = f1_base  - df_outliers['f1'].min()
degradacion_auc_missing  = auc_base - df_missing['auc'].min()
degradacion_f1_missing   = f1_base  - df_missing['f1'].min()

filas_resumen = (
    f'<tr><td style="padding:8px 12px">Ruido gaussiano (50% std)</td>'
    f'<td style="padding:8px 12px;text-align:center">{degradacion_auc_ruido:.4f}</td>'
    f'<td style="padding:8px 12px;text-align:center">{degradacion_f1_ruido:.4f}</td></tr>'
    f'<tr style="background:#f7fafc"><td style="padding:8px 12px">Outliers (30% observaciones)</td>'
    f'<td style="padding:8px 12px;text-align:center">{degradacion_auc_outliers:.4f}</td>'
    f'<td style="padding:8px 12px;text-align:center">{degradacion_f1_outliers:.4f}</td></tr>'
    f'<tr><td style="padding:8px 12px">Missing top {len(features_top)} features</td>'
    f'<td style="padding:8px 12px;text-align:center">{degradacion_auc_missing:.4f}</td>'
    f'<td style="padding:8px 12px;text-align:center">{degradacion_f1_missing:.4f}</td></tr>'
)

contenido = (
    f'<h2 style="color:#2d3748">Fase 6 — M04a · Stress Testing: Robustez del Modelo</h2>'
    + bloque_wilcoxon_html(ROOT, nombre_ganador)
    + '<p style="color:#4a5568;font-size:14px;max-width:800px">'
    f'Evaluación de la robustez del modelo <strong>{nombre_ganador}</strong> ante '
    'perturbaciones controladas. Se simulan tres escenarios adversos: ruido gaussiano '
    'en las features numéricas, presencia de valores extremos y ausencia de las features '
    'más importantes. Un modelo robusto mantiene su rendimiento incluso con datos de baja calidad.'
    '</p>'
    f'<p style="color:#4a5568;font-size:13px">'
    f'Baseline — AUC: <strong>{auc_base:.4f}</strong> | F1: <strong>{f1_base:.4f}</strong>'
    f'</p>'
    '<h3 style="color:#2d3748;margin-top:20px">Degradación máxima por escenario</h3>'
    '<table style="width:60%;border-collapse:collapse;font-size:13px;margin-bottom:24px">'
    '<thead><tr style="background:#edf2f7">'
    '<th style="padding:8px 12px;text-align:left">Escenario</th>'
    '<th style="padding:8px 12px;text-align:center">ΔAUC</th>'
    '<th style="padding:8px 12px;text-align:center">ΔF1</th>'
    '</tr></thead>'
    f'<tbody>{filas_resumen}</tbody></table>'
    + bloque_imagen(img_b64(ruta_ruido),
        'AUC y F1 bajo ruido gaussiano',
        'Degradación progresiva del rendimiento al añadir ruido proporcional '
        'a la desviación estándar de cada feature numérica.')
    + bloque_imagen(img_b64(ruta_outliers),
        'AUC y F1 bajo valores extremos',
        'Impacto de sustituir un porcentaje creciente de observaciones '
        'por valores extremos (media ± 3σ).')
    + bloque_imagen(img_b64(ruta_missing),
        'AUC y F1 bajo features ausentes',
        'Degradación al eliminar progresivamente las features más importantes '
        '(imputadas con la mediana). Muestra cuánto depende el modelo de cada feature.')
    + bloque_imagen(img_b64(ruta_degradacion),
        '🏆 Degradación relativa (% pérdida vs baseline)',
        '% de pérdida de AUC y F1 respecto al baseline. '
        'La línea ámbar marca el umbral del 5% — degradaciones por debajo '
        'indican un modelo robusto apto para producción.')
    + '<div style="margin-top:24px;padding:16px;background:#ebf8ff;'
    f'border-left:4px solid {COLOR_PRIMARIO};border-radius:6px;font-size:13px;color:#2c5282">'
    '<strong>Interpretación:</strong> Una caída de AUC inferior a 0.02 bajo ruido moderado '
    '(10-20% std) indica un modelo robusto. Caídas superiores a 0.05 sugieren '
    'sensibilidad excesiva que debería abordarse antes del despliegue en producción.'
    '</div>'
)

ruta_html = DIR_HTML / 'm04a_stress.html'
render_pagina(
    'f6_m04a_stress.ipynb',
    contenido,
    ruta_html,
    carpeta_notebook='fase6_evaluacion'
)
print(f'✅ HTML generado: {ruta_html}')

✅ HTML generado: C:\FF\AU_UJI_v2\docs\html\fase6\m04a_stress.html
